# Work with Embeddings: Create, Upload, Compare

This notebook explores possibilities to profit from embeddings of a collection of sources. The collection is the digital collection of sources of the project [The School of Salamanca](https://salamanca.school/), and the notebook uses multilingual embeddings from OpenAI, Cohere and, eventually, Ollama.

It makes use of a vectore store API that has been developed for these experiments: the Vector database of the DH at Max Planck Society initiative (dhamps-vdb), developed by [Andreas Wagner](https://www.lhlt.mpg.de/wagner/en) of the [Max Planck Institute for Legal History and Legal Theory](https://www.lhlt.mpg.de/).

## 0. Preliminaries

In [ ]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

In [ ]:
# install packages (if not required by bertopic like pandas, numpy, tqdm, umap-learn, hdbscan etc.)
# !uv pip install asyncio nest_asyncio requests jsonlines logging python-dotenv openai tiktoken cohere cohere-core

# installing bertopic
# !uv pip install bertopic

### 1.2 Load libraries

In [ ]:
import os
import dotenv
import logging
import operator
import itertools
from functools import reduce
import pickle
import jsonlines
from typing import Any, Dict, List
import urllib
import requests
import pandas as pd
import numpy as np
from time import sleep
import asyncio
import nest_asyncio
# from tqdm.asyncio import tqdm
from tqdm.notebook import tqdm
import tiktoken
from openai import AsyncOpenAI
# from umap import UMAP
import umap.umap_ as UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
# from bertopic.backend import OpenAIBackend

### 1.3 Configuration

#### 1.3.1 File paths

In [ ]:
# Input files
file_path_text = './in-data/all_paras.csv'  # text and metadata
file_path_seed_words = './in-data/lemmata_20240322.csv'
file_path_stop_words = './in-data/stopwords-la-new-line_27.01.22.txt'

# Output files
file_path_objects_jsonl = './out-data/output.jsonl'
file_path_embeddings_pickle = "./out-data/embeddings.pkl"
file_path_complete_pickle = "./out-data/everything.pkl"


#### 1.3.2 Limits

In [ ]:
# Here we set the number of documents (paragraphs) to process
# Set it to a lower value until everything runs well, then increase it
# Set it to -1 to process all documents:
max_documents=-1

# Minimum number of tokens for a paragraph to be considered
min_tokens = 10

# We need to heed API call rate limits
# see https://platform.openai.com/account/limits

rate_limit_per_minute = 3000
number_of_concurrent_requests = 30

#### 1.3.3 Parameters for embeddings etc.

In [ ]:
# Embeddings model and parameters
openai_model="text-embedding-3-small" # or text-embedding-3-large
embedding_ctx_limit = 8191            # that's the context window limit for text-embedding-3-small
embedding_encoding = 'cl100k_base'

# Topic modeling parameters
min_topic_size = 20   # how many documents must feature a topic for the topic to be legitimate
top_n_words = 35      # how many words to display in topic representation

#### 1.3.4 Define API services (incl. API keys)

Preparations: do a `cp .env.example .env` in the working directory and edit .env to contain your own OpenAI and DHaMPS-VDB API keys.


In [ ]:
# find the .env file and load it
dotenv.load_dotenv(dotenv.find_dotenv())

# get API keys
api_key_oai = os.getenv("OPENAI_API_KEY") # OpenAI
api_key_vdb = os.getenv("DHAMPS_VDB_API_KEY") # DHaMPS-VDB

# VDB API configuration
API_USER = "sal"
API_PROJECT = "sal-openai-large"
API_PROJECT_ID = 1
API_ENDPOINT = "https://c100-188.cloud.gwdg.de/vdb-api/v1/embeddings/" + API_USER + "/" + API_PROJECT
API_LLM_SERVICE = "openai-large"
HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {api_key_vdb}"}
BATCH_SIZE = None # Set to None to disable batching
RETRY_ATTEMPTS = 3
DELAY_BETWEEN_REQUESTS = 0.2
VERBOSE = False

## 2. Read Input Data

### 2.1 Load data

Then, we read all our documents into a dataframe ...

(TODO: Maybe use [polars](https://pola-rs.github.io/polars-book/) instead of pandas? cf. [blogpost](https://dev.to/ranggakd/polars-vs-pandas-a-brief-tale-of-two-dataframe-libraries-lli))

In [ ]:
# Read csv file with data into a dataframe
with open(file_path_text, 'r') as f:
    print(f"Reading {file_path_text} into a dataframe...")
    df = pd.read_csv(f, index_col='url')


# Add a column specifying the length of the content field
df['len_content'] = df.content.str.len()

### 2.2 Give some information about the data

In [ ]:
print("Shape of dataframe: ")
print(df.shape)
print(f"Number of available documents: {len(df)}\n")
print(f"Dataframe length properties:\n{df.len_content.describe()}")
print("\nFirst rows of dataframe:")
df[:3]

### 2.3 Read seed/stop words

Next, we read our seed words and stop words. For seed words, we take the lemmas of the project dictionary. In fact, the seed words list is an array, each item of which contains an array with potentially multiple, related words.

In [ ]:
with open(file_path_seed_words) as seed_words_file:
    seed_words = [line.rstrip().split(",") for line in seed_words_file]

stop_words = []
with open(file_path_stop_words) as stop_words_file:
    for line in stop_words_file:
        stop_words.append(line.strip())

### 2.4 Give some information about seed/stop words

In [ ]:

print(f"Number of seed words: {len(seed_words)}")
print(seed_words[:10])
print()
print(f"Number of stop words: {len(stop_words)}")
print(stop_words[:10])

## 3. Get Embeddings

### 3.1 Sample data?

Filter too short documents and reduce dataframe to process only {{ max_documents }} docs (for testing). When everything works, we increase this to test for higher loads. In the end, we do it for all the content. The `max_documents` variable is set above, in the [1.3.2. Limits](#132Limits) section.

In [ ]:
if max_documents == -1:
    max_documents = len(df)

# Filter too short paragraphs from the whole input set
tempdocs = df.drop(df[df['content'].map(len) < min_tokens].index)

# Reduce to sample size
docs = tempdocs.iloc[:max_documents].copy()
max_documents, _ = docs.shape
print("Shape of docs dataframe: ")
print(docs.shape)
print("First rows of docs dataframe:")
docs[:3]
print("first 3 ids:")
print(docs.index[:3])

### 3.2 Setup efficient networking

For our API calls, we want to integrate:
- catering for over-long texts (OpenAI API allows us to send a list of inputs, averaging their embeddings)
- caching (store embeddings on disk so we do not have to re-download them every time)
- rate limiting
- asynchronous calls

#### 3.2.1 Saving

First, define functions for saving whatever comes our way...

In [ ]:
def save_dict_pickle(e: dict, path) -> None:
    print("Saving (dictionary) pickle to disk ...")
    with open(path, "wb") as fOut:
        pickle.dump(e, fOut)
    print(f"Saved {len(list(embeddings_dict.keys()))} dictionary entries to {path}\n")

def save_df_pickle(docs: pd.DataFrame, path) -> None:
    print("Saving (dataframe) pickle to disk ...")
    with open(path, "wb") as fOut:
        pickle.dump(docs, fOut)
    print(f"Saved {len(docs)} dataframe entries to {path}\n")

def save_jsonl(json_objects: list, path: str) -> None:
    print("Saving jsonlines to disk ...")
    with jsonlines.open(path, 'w') as fOut:
        fOut.write_all(json_objects)
    print(f"Saved {len(json_objects)} json objects to {path}\n")


#### 3.2.2 Uploading

Define functions to build uploadable data either from dataframe plus embeddings, or from dataframe including embeddings.

In [ ]:
def build_json_object_df(key: str, data: pd.DataFrame) -> dict:
    # Look up records in dictionary and dataframe
    row = df.loc[key]
    # Check if the row was found
    if len(row) == 0:
        print(f"Could not find row in data for key: {key}")
        return None

    embeddings = row["embeddings"]
    # Check if all values in the vector are floats
    # for i, v in enumerate(embeddings):
    #    if not isinstance(v, float):
    #        print(f"Non-float value found at index {i} in key {key}: {v} (type: {type(v)})")
    #        embeddings[i] = float(v)  # Convert to float if necessary

    # Build the json object
    json_object = {
        "text_id": urllib.parse.quote(key, safe=''),
        "user_handle": API_USER,
        "project_handle": API_PROJECT,
        "project_id": API_PROJECT_ID,
        "llm_service_handle": API_LLM_SERVICE,
        "text": row["content"],
        "vector": embeddings,
        "vector_dim": len(embeddings),
        "metadata": {
            "url": key,
            "xmlid": row["xmlid"],
            "wid": row["wid"],
            "author": row["author-name"],
            "author_id": row["author-id"],
            "year": int(row["year"]) if isinstance(row["year"], np.int64) else row["year"],
            "lang": row["lang"],
        }
    }
    return json_object

def build_json_object_df_dict(key: str, data: pd.DataFrame, embeddings_dict: dict) -> dict:
    # Check if dictionary key is available
    if key not in embeddings_dict:
        print(f"Could not find key in embeddings dictionary: {key}")
        return None

    # Look up records in dictionary and dataframe
    row = df.loc[key]
    # Check if the row was found
    if len(row) == 0:
        print(f"Could not find row in data for key: {key}")
        return None

    embeddings = embeddings_dict[key]
    # Check if all values in the vector are floats
    # for i, v in enumerate(embeddings):
    #    if not isinstance(v, float):
    #        print(f"Non-float value found at index {i} in key {key}: {v} (type: {type(v)})")
    #        embeddings[i] = float(v)  # Convert to float if necessary

    # Build the json object
    json_object = {
        "text_id": urllib.parse.quote(key, safe=''),
        "user_handle": API_USER,
        "project_handle": API_PROJECT,
        "project_id": API_PROJECT_ID,
        "llm_service_handle": API_LLM_SERVICE,
        "text": row["content"],
        "vector": embeddings,
        "vector_dim": len(embeddings),
        "metadata": {
            "url": key,
            "xmlid": row["xmlid"],
            "wid": row["wid"],
            "author": row["author-name"],
            "author_id": row["author-id"],
            "year": int(row["year"]) if isinstance(row["year"], np.int64) else row["year"],
            "lang": row["lang"],
        }
    }
    return json_object

def build_json_object_df_ls(key: str, data: pd.DataFrame, embeddings: list) -> dict:
    # Look up records in dictionary and dataframe
    row = df.loc[key]
    # Check if the row was found
    if len(row) == 0:
        print(f"Could not find row in data for key: {key}")
        return None

    # Check if all values in the vector are floats
    # for i, v in enumerate(embeddings):
    #    if not isinstance(v, float):
    #        print(f"Non-float value found at index {i} in key {key}: {v} (type: {type(v)})")
    #        embeddings[i] = float(v)  # Convert to float if necessary

    # Build the json object
    json_object = {
        "text_id": urllib.parse.quote(key, safe=''),
        "user_handle": API_USER,
        "project_handle": API_PROJECT,
        "project_id": API_PROJECT_ID,
        "llm_service_handle": API_LLM_SERVICE,
        "text": row['content'],
        "vector": embeddings,
        "vector_dim": len(embeddings),
        "metadata": {
            "url": key,
            "xmlid": row['xmlid'],
            "wid": row['wid'],
            "author": row['author-name'],
            "author_id": row['author-id'],
            "year": int(row['year']) if isinstance(row['year'], np.int64) else row['year'],
            "lang": row['lang'],
        }
    }
    return json_object

def build_json_objects_df_dict(text_data: pd.DataFrame, embeddings: dict) -> list:
    json_objects = []

    # Iterate over the keys of the embeddings dictionary
    for key, value in tqdm(list(embeddings.items())):
        object = build_json_object_df_dict(key, text_data, embeddings)
        if object is not None:
            json_objects.append(object)
        else:
            print(f"Could not build json object for id: {key}")
            continue

    print("Done. Number of json objects:", len(json_objects))
    return json_objects

def build_json_objects_df(data: pd.DataFrame) -> list:
    json_objects = []

    # Iterate over the keys of the embeddings dictionary
    for id in tqdm(data.index):
        object = build_json_object_df(id, data)
        if object is not None:
            json_objects.append(object)
        else:
            print(f"Could not build json object for id: {id}")
            continue

    print("Done. Number of json objects:", len(json_objects))
    return json_objects


Next, define a function to upload data to our vector store.

In [ ]:
class JsonListApiSender:
    def __init__(
        self,
        api_endpoint: str,
        headers: Dict[str, str] = None,
        retry_attempts: int = 3,
        delay_between_requests: float = 0.1,
        verbose: bool = True
    ):
        """
        Initialize the JSON List API Sender.
        
        Args:
            api_endpoint: URL to send POST requests to
            headers: Optional headers for the request
            retry_attempts: Number of times to retry a failed request
            delay_between_requests: Delay between each request
            verbose: Whether to print detailed logging
        """
        self.api_endpoint = API_ENDPOINT
        self.headers = HEADERS or {"Content-Type": "application/json"}
        self.retry_attempts = RETRY_ATTEMPTS
        self.delay_between_requests = DELAY_BETWEEN_REQUESTS
        self.verbose = VERBOSE
        # self.logger = logging.getLogger(__name__)

    def send_single_item(self, item: Dict[str, Any]) -> bool:
        """
        Send a single JSON item to the API with retry logic.
        
        Args:
            item: JSON object to send
        
        Returns:
            Boolean indicating success or failure
        """
        for attempt in range(self.retry_attempts):
            try:
                response = requests.post(
                    self.api_endpoint,
                    json={"embeddings": [item]},
                    headers=self.headers
                )
                response.raise_for_status()
                
                if self.verbose:
                    print(f"Successfully sent item with text_id: {item.get('text_id')}")
                
                return True
            
            except requests.exceptions.RequestException as e:
                # Log error only for the last attempt
                if attempt == self.retry_attempts - 1:
                    error = e.read() if hasattr(e, 'read') else str(e)
                    try:
                        response
                        try:
                            response.content
                            print(f"Failed to send item. Error: {str(error)}\nResponse: {response.content}\nItem: {item}")
                        except AttributeError:
                            print(f"Failed to send item. Error: {str(error)}\nItem: {item}")

                    except NameError:
                        print(f"Failed to send item. Error: {str(error)}\nItem: {item}")
                
                # Exponential backoff
                sleep(2 ** attempt)
        
        return False

    def send_list(
        self, 
        json_list: List[Dict[str, Any]], 
        num_items: int = BATCH_SIZE
    ) -> Dict[str, int]:
        """
        Send a list of JSON objects to the API.
        
        Args:
            json_list: List of JSON objects to send
            num_items: Number of items to send (None = send all)
        
        Returns:
            Dictionary with counts of successful and failed items
        """
        # Determine number of items to send
        if num_items is None:
            num_items = len(json_list)
        else:
            num_items = min(num_items, len(json_list))
        
        # Slice the list to the desired number of items
        items_to_send = json_list[:num_items]
        
        # Tracking variables
        success_count = 0
        fail_count = 0
        failed_items = set()
        
        # Log start of sending process
        print(f"Starting to send {num_items} items to {self.api_endpoint}")
        
        # Progress bar
        for item in tqdm(items_to_send, desc="Sending items", total=num_items):
            if self.send_single_item(item):
                success_count += 1
            else:
                fail_count += 1
                failed_items.add(item.get('text_id'))
            
            # Delay between requests
            sleep(self.delay_between_requests)
        
        # Log overall results
        self.logger.info(
            f"Sending complete. "
            f"Sent: {num_items}, "
            f"Successful: {success_count}, "
            f"Failed: {fail_count}"
        )
        
        return {
            "total_sent": num_items,
            "successful": success_count,
            "failed": fail_count,
            "failed_ids": list(failed_items)
        }

# Example usage:
# sender = JsonListApiSender(api_endpoint="https://your-api-endpoint.com", verbose=True)
# results = sender.send_list(your_json_list, num_items=10)

Initialize sender

In [ ]:
sender = JsonListApiSender(api_endpoint=API_ENDPOINT, verbose=VERBOSE)

#### 3.2.3 Manage embeddings creating

Next, we define a function to separate text that is too long into chunks. It returns either a list of token ids or - if too long - a list of lists of token ids. (In this case, their embeddings will be averaged below.) See [OpenAI Cookbook on Embedding long inputs](https://cookbook.openai.com/examples/embedding_long_inputs).

In [ ]:
# batched (a function from Python's own cookbook) breaks up a sequence into chunks
def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(itertools.islice(it, n))):
        yield batch

# chunked_tokens encodes a string into tokens and then breaks it up into chunks
def chunked_tokens(text: str) -> list:
    encoding = tiktoken.encoding_for_model(openai_model)
    tokens = encoding.encode(text)
    chunks_iterator = list(batched(tokens, embedding_ctx_limit))
    yield from chunks_iterator

### 3.3 Acquire embeddings



Define a function to asynchronously create embeddings for texts or chunks of texts, but then wrap it in a caching function, so we don't have to re-generate embeddings all the time.

When creating embeddings, also create a JSON object with all available data for the text and push it to the vector store. Build a list of ids that were uploaded in order to be able to later upload those that weren't uploaded during creation but just loaded from the cache file.

In [ ]:
delay = number_of_concurrent_requests * 60.0 / rate_limit_per_minute
uploaded_ids = {}
json_objects = []

nest_asyncio.apply()

async def retrieve_embeddings(client: AsyncOpenAI, sem: asyncio.Semaphore, index: str, row, delay=delay) -> dict:
    """Make an async embedding call. If text is too long. Make several calls and average results."""
    async with sem:
        await asyncio.sleep(delay)
    
        # If I understand correctly, the OpenAI API already provides one averaged set of embeddings,
        # even if a list of inputs has been provided. So we do not need to do the averaging ourselves...
        # the lines relevant to averaging are thus commented out.
    
        # chunk_embeddings = []
        # chunk_lens = []
    
        # for chunk in chunked_tokens(row['content']):
            # print(chunk[:3])
            # response = await client.embeddings.create(input=chunk, model=openai_model)
            # chunk_embeddings.append(response.data[0].embedding)
            # chunk_lens.append(len(chunk))
    
        # chunk_embeddings = np.average(chunk_embeddings, axis=0, weights=chunk_lens)
        # chunk_embeddings = chunk_embeddings / np.linalg.norm(chunk_embeddings)  # normalizes length to 1
        # return { index: chunk_embeddings.tolist() }

        input = chunked_tokens(row['content'])
        response = await client.embeddings.create(input=input, model=openai_model)
        embeddings = response.data[0].embedding

        # Directly upload to vector database
        new_object = build_json_object_df_ls(index, docs, embeddings)
        upload_success = sender.send_single_item(new_object)
        if upload_success:
            uploaded_ids[index] = True
            json_objects.append(new_object)
        else:
            print(f"Failed to upload {index} to vector database.")
        return { index: embeddings }

async def control_embeddings_retrieval(client: AsyncOpenAI, sem: asyncio.Semaphore, dfi) -> dict:
    """Control async embedding calls."""
    print("Creating embeddings. This can take a while ...")
    tasks = [asyncio.create_task(retrieve_embeddings(client, sem, index, row)) for index, row in dfi.iterrows()]
    # results = await tqdm.gather(*tasks)
    results = [await f for f in tqdm(asyncio.as_completed(tasks), total=len(tasks))]
    # we combine the results of the various API calls with the ior ("|") operator
    return reduce(operator.ior, results)

async def get_embeddings(client: AsyncOpenAI, sem: asyncio.Semaphore, dfi) -> dict:
    """Get embeddings: either by reading from cache or by making API calls."""
    if not os.path.exists(file_path_embeddings_pickle):
        # There is no embeddings file
        # → create embeddings
        embeddings = await control_embeddings_retrieval(client, sem, dfi)
        return embeddings

    else:
        # There is an embeddings file
        # → read it
        print("Embeddings found.", end="")
        with open(file_path_embeddings_pickle, "rb") as fIn:
            cache_data = pickle.load(fIn)
            if openai_model in cache_data:
                print(" Model found.", end="")
                if len(cache_data[openai_model]) >= max_documents:
                    print(" Sufficient number of embeddings found. Reading ...", end="")
                    embeddings = cache_data[openai_model]
                    return embeddings
                else:
                    print("\nWarning: Fewer " + openai_model + " embeddings found in cache (" + str(len(cache_data[openai_model])) + ") than needed for " + str(max_documents) + " documents.")
                    current_embeddings = await control_embeddings_retrieval(client, sem, dfi)
                    all_embeddings = cache_data | { openai_model: current_embeddings } 
                    # store embeddings
                    save_pickle(all_embeddings, file_path_embeddings_pickle)
                    return current_embeddings
            else:
                # While we do have embeddings, they don't fit the current embedding model.
                # → create embeddings for the current model config.
                print("\nWarning: No " + openai_model + " embeddings found in cache.")
                current_embeddings = await control_embeddings_retrieval(client, sem, dfi)
                all_embeddings = cache_data | { openai_model: current_embeddings } 
                # store embeddings
                save_pickle(all_embeddings, file_path_embeddings_pickle)
                return current_embeddings

Create request client and semaphore, then call the embeddings getter function - either load from pickle file if one is present, or create new embeddings by calling APIs.

In [ ]:
oai_async_client = AsyncOpenAI(api_key=api_key_oai)
sem = asyncio.Semaphore(number_of_concurrent_requests)

# Use the cache-capable asynchronous "get_embeddings" method on our to-be-processed docs
# Store the returned result in a new dict "embeddings_dict"
embeddings_dict = asyncio.run(get_embeddings(oai_async_client, sem, docs))
apiDescriptionUrl="/openapi.yaml"
print("First 5 embeddings' keys:\n")
print(list(embeddings_dict.keys())[:5])

### 3.4 Merge Embeddings and Dataframe

Merge the embeddings into our dataframe - by row id (which is the paragraph url).

In [ ]:
docs['embeddings'] = docs.index.map(embeddings_dict)

# Display the DataFrame
print("\nFirst rows of docs dataframe:")
docs[:3]

### 3.5 Save Data

Save embeddings and the data that now includes embeddings.

In [ ]:
# Save the embeddings dictionary to disk
print(f"Saving {file_path_embeddings_pickle} to disk ...")
save_dict_pickle({ openai_model: embeddings_dict }, file_path_embeddings_pickle)

# Save the complete dataframe to disk
print(f"Saving {file_path_complete_pickle} to disk ...")
save_df_pickle(docs, file_path_complete_pickle)

# Save the jsonlines to disk
print(f"Saving {file_path_objects_jsonl} to disk ...")
save_jsonl(json_objects, file_path_objects_jsonl)

### 3.6 Postliminary: Isolate some types and give stats

Do we need this?: Prepare data objects with only a part of the data - text paragraphs and embeddings themselves.

In [ ]:
paragraphs = docs['content']
embeddings = np.array(list(docs['embeddings']))

In [ ]:
print(len(paragraphs))
print(len(embeddings))
print(len(embeddings[0]))
print(type(embeddings))

## 4. Upload Data to Vector Store

For those records that have not been uploaded yet, do it now.

In [ ]:
# for embeddings entries that have not yet been uploaded, build json objects and upload them
unuploaded_dict = {k: v for k, v in embeddings_dict.items() if k not in uploaded_ids}
upload_objects = build_json_objects_df_dict(docs, unuploaded_dict)
sender.send_list(upload_objects)

## 5. Fit (Train) Topic Model

We split the remaining BERTopic steps between those that are part of the preparation of the (training of the) topic model, and those that affect the topic representationse: Dimensionality Reduction and Clustering on the one hand, Vectorization, c-TF-IDF representation on the other.

For all of these, we supply the default algorithms, but establish code to be able to easily change any of them in the future.

In [ ]:
# Define BERTopic parameters
# For helpful comments about the parameters,
# see https://medium.com/grabngoinfo/hyperparameter-tuning-for-bertopic-model-in-python-104445778347

# A. Dimensionality Reduction: umap_model

# Initialize the UMAP model for dimensionality reduction
dimred_model = umap.UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine', random_state=42)
# dimred_model = umap.UMAP()

# If we should want to not do any dimensionality reduction, we can use this:
# from bertopic.dimensionality import BaseDimensionalityReduction
# dimred_model = BaseDimensionalityReduction()

# B. Clustering

cluster_model = HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True)


Do the actual BERTopic model initialization and fitting:

In [ ]:
topic_model = BERTopic(
    umap_model=dimred_model,       # yes, the parameter name is called like the default method
    hdbscan_model=cluster_model,   # yes, the parameter name is called like the default method
    min_topic_size=min_topic_size,
    top_n_words=top_n_words,
    calculate_probabilities=True,
    verbose=True
)

tm_loaded = False
if os.path.isdir(file_path_tm_cache):
    try:
        topic_model = BERTopic.load(file_path_tm_cache)
    except Exception as e:
        print("Could not load topic model although a cache dir is present. Error: " + e)
        print("Fitting new topic model instead...")
        topics, probs = topic_model.fit_transform(paragraphs, embeddings)
    else:
        topics, probs = topic_model.transform(paragraphs, embeddings)
        tm_loaded = True
else:
    topics, probs = topic_model.fit_transform(paragraphs, embeddings)

print("\n")

### Interpret results

Get some diagnostic sample from the topics and probabilites - first, without finetuning the topic representation.

In [ ]:
topics_count = len(topic_model.topic_labels_)
topics_count

In [ ]:
df_topic_freq = topic_model.get_topic_freq()
df_topic_freq

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)

In [ ]:
topic_model.get_document_info(paragraphs)

## Tune Topic Representations

Now, if the above looks halfway reasonable, continue with work on improving the topic representation. This should work without retraining the model ...

### Vectorizer & c-TF/IDF Parameters

First, define vectorizer and tf/idf models, incorporating stopwords and seed words lists:

In [ ]:
# Topic representation (a): Vectorization

# Again, for helpful comments on parameters,
# see https://medium.com/grabngoinfo/hyperparameter-tuning-for-bertopic-model-in-python-104445778347

# min_df (minimum document frequency): in how many documents must a word occur for it to be eligible for topic representation
# max_features: how large is the vocabulary of words to consider

from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words=stop_words, ngram_range=(1, 3), min_df=6, max_features=100_000)

In [ ]:
# Topic representation (b): c-TF-IDF representation

# seed_multiplier: what to multiply the TF/IDF score of a word with when it is mentioned in the seed word list

from bertopic.vectorizers import ClassTfidfTransformer
ctfidf_model = ClassTfidfTransformer(seed_words=seed_words, seed_multiplier=10, reduce_frequent_words=True)

### Topic Labels and Descriptions

In [ ]:
# find the .env file and load it
load_dotenv(find_dotenv())

# get our API keys
oai_api_key = getenv("OPENAI_API_KEY")
coh_api_key = getenv("COHERE_API_KEY")

# KeyBERT
from bertopic.representation import KeyBERTInspired
keybert_model = KeyBERTInspired()

# OpenAI Labels
from bertopic.representation import OpenAI
import openai
oai_client = openai.OpenAI(api_key=oai_api_key)
oai_label_model = OpenAI(oai_client, model="gpt-4o", delay_in_seconds=10, chat=True)

# OpenAI Summaries
summarization_prompt = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a summarizing description of this topic:

"""
oai_sum_model = OpenAI(oai_client, model="gpt-4o", chat=True, prompt=summarization_prompt, nr_docs=5, delay_in_seconds=10)

# Cohere
import cohere
from bertopic.representation import Cohere
coh_client = cohere.Client(coh_api_key)
coh_label_model = Cohere(coh_client, model="command-r-plus", delay_in_seconds=15)

# Cohere Summaries
coh_sum_model = Cohere(coh_client, model="command-r-plus", prompt=summarization_prompt, nr_docs=5, delay_in_seconds=20)

representation_model = {
    "Main":  oai_label_model,
    "Cohere-Label": coh_label_model,
    # "KeyBERT": keybert_model,
    "GPT-Summary":  oai_sum_model,
    # "Cohere-Summary": coh_sum_model
}

In [ ]:
if not tm_loaded:
    topic_model.update_topics(paragraphs, vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model, top_n_words=top_n_words, representation_model=representation_model)

### Lemmatizing?

Something to try out. It's not clear how much improvement it brings. It's clear however, that it has to come after creating the embeddings. And if we want to create labels or summaries with a LLM, we need to do it before lemmatizing everything, too...

In [ ]:
# TODO:
# You can hook into Vectorization to do lemmatiziation:
# cf. https://github.com/MaartenGr/BERTopic/issues/286
#
# class LemmaTokenizer:
#    def __init__(self):
#        self.wnl = WordNetLemmatizer()
#    def __call__(self, doc):
#        return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]
#
# vectorizer_model= CountVectorizer(tokenizer=LemmaTokenizer())
#
# topic_model = BERTopic(vectorizer_model=vectorizer_model)

## Interpret (new) results

Now, get new diagnostics.

In [ ]:
df_topic_freq = topic_model.get_topic_freq()
df_topic_freq

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)

In [ ]:
if not os.path.exists('./figs'):
    os.makedir('./figs')

In [ ]:
fig_barchart = topic_model.visualize_barchart(top_n_topics=topics_count, n_words=8, custom_labels=True)
fig_barchart.write_html("figs/fig_barchart.html")
# fig_barchart

In [ ]:
fig_topics = topic_model.visualize_topics()
fig_topics.write_html("figs/fig_topics.html")
# fig_topics

In [ ]:
# Topic Similarity Matrix
fig_heatmap = topic_model.visualize_heatmap(custom_labels=True)
fig_heatmap.write_html("figs/fig_heatmap.html")
# fig_heatmap

In [ ]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(paragraphs, reduced_embeddings=reduced_embeddings)

In [ ]:
# Check if have a good cut-off of top_n_words
fig_termrank = topic_model.visualize_term_rank()
fig_termrank.write_html("figs/fig_termrank.html")
# fig_termrank

## Save results

In [ ]:
# Export our topics to a new csv file, too
topic_model.get_topic_info().to_csv(file_path_tm_export, index=False)

# And save the topic model with all sub-models to a pickle file
print("Saving Topic Model to disk ...")
topic_model.save(file_path_tm_cache, serialization="safetensors", save_ctfidf=True, save_embedding_model=False)

# with open(file_path_tm_cache + ".pkl", "wb") as fOut:
#    pickle.dump(topic_model, fOut)
# print("Done.")

In [ ]:
# Save the generated topics for all the documents to the dataframe
docs['topic'] = topic_model.topics_

In [ ]:
# Display the DataFrame
print("\nFirst rows of docs dataframe:")
docs[:3]

In [ ]:
# Export the documents data frame to a new csv file
docs.to_csv(file_path_export, index=False)

## Next steps

See the discussion in this cookbook: [Nick T.: "Topic Modeling with BERTopic: A Cookbook with an End-to-end Example (Part 2)"](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-2-1ae591f76a25).

<div class="alert alert-warning"><b>After working and getting good results in the following steps, do not forget to update the data frame and save it to CSV again.</b></div>

### Topics per Class (author, year etc.)

#### authors

In [ ]:
topics_per_author = topic_model.topics_per_class(paragraphs, classes=docs['author'], global_tuning=False)
fig_tpa = topic_model.visualize_topics_per_class(topics_per_author, top_n_topics=6, normalize_frequency=True, custom_labels=True)
fig_tpa.write_html("figs/fig_tpa.html")
# fig_tpa

#### years

In [ ]:
topics_per_year = topic_model.topics_per_class(paragraphs, classes=docs['year'], global_tuning=False)
fig_tpy = topic_model.visualize_topics_per_class(topics_per_year, top_n_topics=6, normalize_frequency=True, custom_labels=True)
fig_tpy.write_html("figs/fig_tpy.html")
# fig_tpy

### Outlier Reduction?

see [Outliers Reduction](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-1-3ef739b8d9f8#8aaf) in the cookbook.

In [ ]:
# rinse and repeat

### ("Manual") Topic Merging

In [ ]:
# are there topics we would prefer to have merged? It goes like this:
# topics_to_merge = [
#    [10, 11]
# ]
# topic_model.merge_topics(paragraphs, topics_to_merge)

### Understand why a particular document is assigned to its topic

...

In [ ]:
topic_distr, topic_token_distr = topic_model.approximate_distribution(paragraphs, calculate_tokens=True, use_embedding_model=True)

In [ ]:
example_doc_id = 426
fig_ad = topic_model.visualize_approximate_distribution(paragraphs[example_doc_id], topic_token_distr[example_doc_id])
fig_ad.write_html("figs/fig_ad.html")
# fig_ad

In [ ]:
# To visualize the topic distributions in a document
# topic_model.visualize_distribution(topic_distr[example_doc_id], min_probability=0.00002)
# Visualize probability distribution
fig_dist = topic_model.visualize_distribution(topic_model.probabilities_[example_doc_id], min_probability=0.015, custom_labels=True)
fig_dist.write_html("figs/fig_dist.html")
# fig_dist

In [ ]:
# Topic Hierarchy
fig_hier = topic_model.visualize_hierarchy()
fig_hier.write_html("figs/fig_hier.html")
# fig_hier

Either assign labels "manually" by interpreting them intellectually. Or ask ChatGPT (or another LLM) to produce a label and description based on the most salient words and documents...

In [ ]:
# Add topic labels and descriptions
topic_labels_dict = {}
topic_descriptions_dict = {}
docs['topic_label'] = docs.topic.map(topic_labels_dict)
docs['topic_description'] = docs.topic.map(topic_descriptions_dict)

## Dynamic Topic Modeling

Next thing to try: [*dynamic* topic modeling](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)

In [ ]:
years = docs['year']
topics_over_time = topic_model.topics_over_time(paragraphs, years, evolution_tuning=True, global_tuning=True)

In [ ]:
fig_tot = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)
fig_tot.write_html("figs/fig_tot.html")
# fig_tot

You can also visualize some concrete topics:

In [ ]:
fig_tot_9_10_72 = topic_model.visualize_topics_over_time(topics_over_time, topics=[9, 10, 72])
fig_tot_9_10_72.write_html("figs/fig_tot_9_10_72.html")
# fig_tot_9_10_72